# PHIA 单 Agent 基线

本 Notebook 演示 PHIA（Personal Health Insights Agent）基线，即 PHA 研究用来对照的原始单 Agent 架构。

## 研究问题

专用 Agent 分工能否优于一个通用 Agent？

## 架构对比

| 比较项 | PHIA 单 Agent 基线 | PHA 多 Agent 架构 |
| --- | --- | --- |
| Agent | 1 个通用 Agent | 3 个专用 Agent 和 1 个编排器 |
| 路由 | 由单个 Agent 处理 | 按问题选择 Agent |
| 提示词 | 1 套综合提示词 | 4 套各自负责不同任务的提示词 |
| 工具 | 统一工具池 | 各 Agent 使用相应工具 |

## PHIA 如何工作

用户提问后，单个 ReAct Agent 根据综合前置提示、DataFrame 字段结构和健康背景组织回答。它可以使用 `tool_code`（Python 和 pandas）、`search`（网页搜索）和 `finish`（提交最终答复）。

## 环境搭建

In [ ]:
import os
import sys

# 设置 API 密钥( 替换为您自己的或设定的环境变量)
# os.environ ['GEMINI_API_KEY']="这里的钥匙"
# os.environ ['TAVILY_API_KEY'] = '你的钥匙 -这里'#可选，用于网页搜索

# 如果运行在笔记本文件夹中，将父目录添加到路径
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

# 请检查access-date=中的日期值 (帮助)
from pha.agents import is_onetwo_available

if not is_onetwo_available():
    print("⚠️ PHIA 基线需要安装 OneTwo")
    print("请按 README 运行 setup.sh 安装 OneTwo")
else:
    print("✓ OneTwo 可用")

In [ ]:
# API 密钥
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY') or input('请输入 Gemini API 密钥：')
TAVILY_API_KEY = os.environ.get('TAVILY_API_KEY') or input('请输入 Tavily API 密钥（选填）：') or None

print(f"Gemini API 密钥： {'✓ 已设置' if GEMINI_API_KEY else '✗ 未设置'}")
print(f"Tavily API 密钥： {'✓ 已设置' if TAVILY_API_KEY else '✗ 未设置（网页搜索不可用）'}")

In [ ]:
# 装入设置和数据
from config.settings import Settings

settings = Settings()
print(f"数据目录： {settings.data_dir}")
print(f"健康摘要路径： {settings.summary_path}")

## 创建 PHIA 基线

In [ ]:
from pha.agents import create_phia_baseline

# PHIA 少样本示例的路径
# 这些是最初的PHIA实例，教特工如何回应
FEW_SHOT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'few_shots', 'phia')
if not os.path.exists(FEW_SHOT_DIR):
    FEW_SHOT_DIR = os.path.join(os.getcwd(), 'few_shots', 'phia')

print(f"少样本目录： {FEW_SHOT_DIR}")
print(f"可用的少样本 Notebook 数量： {len([f for f in os.listdir(FEW_SHOT_DIR) if f.endswith('.ipynb')]) if os.path.exists(FEW_SHOT_DIR) else 0}")

In [ ]:
# 创建 PHIA基线
# NOTE: PHIA 使用 Gemini 2.5 默认(匹配原始执行) pro.
# 用于基线比较目的

baseline = create_phia_baseline(
    gemini_api_key=GEMINI_API_KEY,
    tavily_api_key=TAVILY_API_KEY,
    settings=settings,
    few_shot_dir=FEW_SHOT_DIR,
    debug_verbose=True,
)

print(f"\n✓ 已创建 PHIA 基线")
print(f"  模型：{baseline._model_name}")
print(f"  少样本示例：{len([f for f in os.listdir(FEW_SHOT_DIR) if f.endswith('.ipynb')])} 本 Notebook")

## 测试问题

尝试让 PHIA 处理不同类型的问题。

### 问题 1：睡眠分析（数据和解读）

In [ ]:
query1 = "过去两周我的睡眠怎么样？深睡时间够吗？"

print(f"问题：{query1}")
print("="*60)

response1 = baseline.respond(query1)
print("\n回复：")
print(response1)

### 问题 2：活动分析

In [ ]:
query2 = "我最常做哪些运动？哪些运动消耗的热量最多？"

print(f"问题：{query2}")
print("="*60)

response2 = baseline.respond(query2)
print("\n回复：")
print(response2)

### 问题 3：健康建议（需要网页搜索）

In [ ]:
query3 = "结合我的数据，我可以怎样改善心脏健康？"

print(f"问题：{query3}")
print("="*60)

response3 = baseline.respond(query3)
print("\n回复：")
print(response3)

## 对比要点

| 比较项 | PHIA | PHA |
| --- | --- | --- |
| 回答质量 | 由单个 Agent 给出 | 汇总多个 Agent 的分析 |
| 数据分析 | 通用 Agent 分析 | 数据科学 Agent 专门分析 |
| 医学背景 | 基本的背景信息 | 领域专家 Agent 解读 |
| 可执行建议 | 通用建议 | 健康教练 Agent 给出建议 |
| 执行方式 | 每次使用同一套处理流程 | 按问题路由至相关 Agent |

## 清理

In [ ]:
# 重置对讲状态
baseline.reset_conversation()
print("对话状态已重置。")